# MNIST + ANN + Keras Callbacks

강의 예제 흐름: **라이브러리 → MNIST → Sequential ANN → compile → Callbacks(ModelCheckpoint, TensorBoard, LearningRateScheduler, CSVLogger) → fit → evaluate → TensorBoard**.

필기: [`08.keras_callbacks.md`](08.keras_callbacks.md) · Colab 예제 링크는 필기에 정리됨.

---

## 라이브러리 import

In [ ]:
import tensorflow as tf

## MNIST 데이터 준비

In [ ]:
mnist = tf.keras.datasets.mnist

# set mnist train and test data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

## ANN 모델 설정

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10),
])

## Loss Function & Optimizer 설정

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

## Callbacks 설정

- **ModelCheckpoint**: 에폭마다(기본) 가중치/모델 저장 경로 지정
- **TensorBoard**: `./logs` 에 스칼라 등 기록
- **LearningRateScheduler**: 아래 `scheduler` — 처음 몇 에폭은 LR 유지, 이후 매 에폭 `exp(-0.1)` 배로 감소 (강의 코드는 `epoch < 5` 기준)
- **CSVLogger**: `training.log` 에 에폭별 메트릭 저장

In [ ]:
import math

checkpoint_filepath = 'mnist_ann_model.h5'
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_filepath)

tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="./logs")

# 처음 5 epoch는 학습률 유지, 이후 매 epoch마다 lr * exp(-0.1)
def scheduler(epoch, lr):
    if epoch < 5:
        return float(lr)
    return float(lr) * math.exp(-0.1)

learning_rate_callback = tf.keras.callbacks.LearningRateScheduler(scheduler)
csv_logger = tf.keras.callbacks.CSVLogger('training.log')

## Training

In [ ]:
EPOCHS = 10

model.fit(x_train, y_train, epochs=EPOCHS,
          callbacks=[
              model_checkpoint_callback,
              tensorboard_callback,
              learning_rate_callback,
              csv_logger,
          ])

## Evaluate

In [ ]:
model.evaluate(x_test, y_test, verbose=2)

## TensorBoard (Jupyter / Colab)

로컬 Jupyter에서는 아래 매직이 동작하려면 `tensorboard` 패키지가 필요합니다. 안 되면 터미널에서 `tensorboard --logdir=logs` 후 브라우저에서 `http://localhost:6006` 을 엽니다.

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs